# Sri Lanka crop requirements vs. NASA POWER conditions
Planning aid only. Most ranges in the CSV are **estimates** (not verified DOA thresholds), and satellite/reanalysis values are regional context, not field measurements. Local expertise must validate results.

Run the cells top to bottom. You will upload `sri_lanka_crop_cultivation_requirements.csv` in step 1.

## 1. Load the CSV

In [ ]:
import io, re
import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    from google.colab import files
    up = files.upload()                       # choose the crop CSV
    df = pd.read_csv(io.BytesIO(next(iter(up.values()))))
except ImportError:                           # running outside Colab
    df = pd.read_csv("sri_lanka_crop_cultivation_requirements.csv")

print(df.shape)
df[["crop_name", "crop_group", "optimal_temperature_C", "seasonal_rainfall_mm"]].head()

## 2. Parse the range text
Values look like `"15-24 (ESTIMATE; cool conditions preferred)"`. We extract min and max, and flag whether the matching `*_status` column says **DOA recommendation** (verified) or an estimate.

In [ ]:
def parse_range(text):
    s = str(text).replace(",", "")
    m = re.search(r"(\d+\.?\d*)\s*-\s*(\d+\.?\d*)", s)
    if m:
        return float(m[1]), float(m[2])
    m = re.search(r"(\d+\.?\d*)", s)          # e.g. "About 5.5"
    return (float(m[1]), float(m[1])) if m else (np.nan, np.nan)

VARS = {  # key: (label, value column, status column)
    "temp": ("Temperature (°C)",        "optimal_temperature_C",     "temperature_status"),
    "rain": ("Seasonal rainfall (mm)",  "seasonal_rainfall_mm",      "rainfall_status"),
    "ph":   ("Soil pH",                 "preferred_soil_pH",         "soil_pH_status"),
    "rh":   ("Relative humidity (%)",   "relative_humidity_percent", "humidity_status"),
}

crops = df[["crop_name", "crop_group", "water_requirement"]].copy()
for k, (label, col, st) in VARS.items():
    rng = df[col].apply(parse_range)
    crops[f"{k}_min"] = [r[0] for r in rng]
    crops[f"{k}_max"] = [r[1] for r in rng]
    crops[f"{k}_doa"] = df[st].astype(str).str.startswith("DOA")

# How much of each field is actually DOA-verified?
print(crops[[f"{k}_doa" for k in VARS]].sum().rename(lambda c: c.replace("_doa", "")).to_string())

## 3. Fetch NASA POWER daily data for your location
Edit the settings. `T2M` = air temperature at 2 m, `RH2M` = relative humidity, `PRECTOTCORR` = bias-corrected precipitation (a MERRA-2-based POWER product, **not** raw GPM IMERG). POWER is gridded (about 0.5° resolution), so values describe the surrounding area, not your exact field.

In [ ]:
# ---------- EDIT THESE ----------
LAT, LON     = 7.29, 80.63        # example: Kandy area; replace with your field location
START, END   = 2001, 2025         # full calendar years
SEASON_MONTHS = [10, 11, 12, 1, 2, 3]   # Maha ~Oct-Mar; use [4,5,6,7,8,9] for Yala, or your crop's window
PH_BAND      = (5.5, 6.5)         # from your own soil test
# --------------------------------

url = "https://power.larc.nasa.gov/api/temporal/daily/point"
params = dict(parameters="T2M,RH2M,PRECTOTCORR", community="AG",
              latitude=LAT, longitude=LON,
              start=f"{START}0101", end=f"{END}1231", format="JSON")
r = requests.get(url, params=params, timeout=120)
r.raise_for_status()
p = r.json()["properties"]["parameter"]

w = pd.DataFrame(p)
w.index = pd.to_datetime(w.index, format="%Y%m%d")
w = w.replace(-999, np.nan)                     # POWER fill value
w.columns = ["temp", "rh", "rain"]
w = w.rename_axis("date")
print(w.describe().round(1))

## 4. Turn the daily data into a growing-window band (P10–P90)
* **Temperature and humidity:** 10th–90th percentile of daily values inside the chosen months.
* **Seasonal rainfall:** total rainfall for each season, then the 10th–90th percentile across years. Seasons that wrap the new year (like Maha) are kept together, and incomplete seasons are dropped.

In [ ]:
win = w[w.index.month.isin(SEASON_MONTHS)].copy()

# Assign each day to a season-year (so Oct-Dec 2010 and Jan-Mar 2011 are one Maha season)
start_m = SEASON_MONTHS[0]
win["season_year"] = win.index.year - (win.index.month < start_m).astype(int) \
                     if SEASON_MONTHS[0] > SEASON_MONTHS[-1] else win.index.year

# keep only complete seasons
n_days = win.groupby("season_year")["rain"].count()
full = n_days[n_days >= 0.95 * n_days.max()].index
seasonal_rain = win[win.season_year.isin(full)].groupby("season_year")["rain"].sum()

p10_90 = lambda s: (float(np.nanpercentile(s, 10)), float(np.nanpercentile(s, 90)))
BAND = {
    "temp": p10_90(win["temp"]),
    "rain": p10_90(seasonal_rain),
    "ph":   PH_BAND,
    "rh":   p10_90(win["rh"]),
}
for k, v in BAND.items():
    print(f"{VARS[k][0]:26s} {v[0]:8.1f} - {v[1]:8.1f}")
print(f"Seasons used for rainfall: {len(seasonal_rain)}")

## 5. Fit per variable
* **within** – your band lies fully inside the crop's range
* **partial** – the two overlap
* **outside** – no overlap
* **no data** – missing value

There is deliberately **no single combined score**.

In [ ]:
def fit(k, row):
    lo, hi = row[f"{k}_min"], row[f"{k}_max"]
    b0, b1 = BAND[k]
    if np.isnan(lo) or np.isnan(b0):
        return "no data"
    if b0 >= lo and b1 <= hi:
        return "within"
    return "partial" if min(b1, hi) >= max(b0, lo) else "outside"

for k in VARS:
    crops[f"{k}_fit"] = crops.apply(lambda r: fit(k, r), axis=1)

pts = {"within": 2, "partial": 1, "outside": 0, "no data": 0}
crops["fit_points"] = sum(crops[f"{k}_fit"].map(pts) for k in VARS)   # used for sorting only
crops = crops.sort_values(["fit_points", "crop_name"], ascending=[False, True]).reset_index(drop=True)
crops[["crop_name"] + [f"{k}_fit" for k in VARS]].head(15)

## 6. Range-bar chart (main view)
Solid bars = DOA-verified range. Hatched bars = estimate. The orange band = your conditions from NASA POWER (and your soil test for pH).

In [ ]:
order = crops["crop_name"].tolist()[::-1]            # best fit at top
fig = make_subplots(rows=1, cols=4, shared_yaxes=True, horizontal_spacing=0.015,
                    subplot_titles=[VARS[k][0] for k in VARS])

for i, k in enumerate(VARS, start=1):
    span = np.nanmax(crops[f"{k}_max"]) - np.nanmin(crops[f"{k}_min"])
    for doa, name, shape in [(True, "DOA-verified", ""), (False, "Estimate", "/")]:
        d = crops[crops[f"{k}_doa"] == doa].dropna(subset=[f"{k}_min"])
        width = (d[f"{k}_max"] - d[f"{k}_min"]).clip(lower=span * 0.01)   # keep single values visible
        fig.add_trace(go.Bar(
            y=d["crop_name"], x=width, base=d[f"{k}_min"], orientation="h",
            name=name, legendgroup=name, showlegend=(i == 1),
            marker=dict(color="#2f6f4f" if doa else "#8aa89a",
                        pattern=dict(shape=shape, size=5, bgcolor="rgba(0,0,0,0)")),
            customdata=np.c_[d[f"{k}_min"], d[f"{k}_max"]],
            hovertemplate="%{y}: %{customdata[0]} to %{customdata[1]}<extra>" + name + "</extra>"),
            row=1, col=i)
    b0, b1 = BAND[k]
    fig.add_vrect(x0=b0, x1=b1, fillcolor="orange", opacity=0.25, line_width=1,
                  line_color="darkorange", row=1, col=i)

fig.update_yaxes(categoryorder="array", categoryarray=order, tickfont_size=10)
fig.update_layout(height=max(500, 20 * len(crops)), barmode="overlay", bargap=0.25,
                  title=f"Crop requirement ranges vs. conditions at {LAT}, {LON} "
                        f"(months {SEASON_MONTHS}, {START}-{END})",
                  legend=dict(orientation="h", y=1.06), template="plotly_white",
                  margin=dict(l=150, r=20, t=100, b=40))
fig.show()

## 7. Fit matrix (crop × variable)

In [ ]:
codes = {"outside": 0, "partial": 1, "within": 2, "no data": 3}
Z   = np.array([[codes[crops.loc[i, f"{k}_fit"]] for k in VARS] for i in crops.index])
TXT = np.array([[crops.loc[i, f"{k}_fit"] for k in VARS] for i in crops.index])

colors = ["#b23a3a", "#e5b45a", "#2f6f4f", "#cccccc"]
scale = [[0, colors[0]], [.25, colors[0]], [.25, colors[1]], [.5, colors[1]],
         [.5, colors[2]], [.75, colors[2]], [.75, colors[3]], [1, colors[3]]]

hm = go.Figure(go.Heatmap(z=Z, x=[VARS[k][0] for k in VARS], y=crops["crop_name"],
                          text=TXT, texttemplate="%{text}", colorscale=scale,
                          zmin=0, zmax=3, showscale=False, xgap=2, ygap=2,
                          hovertemplate="%{y} | %{x}: %{text}<extra></extra>"))
hm.update_yaxes(autorange="reversed", tickfont_size=10)
hm.update_xaxes(side="top")
hm.update_layout(height=max(500, 20 * len(crops)), template="plotly_white",
                 margin=dict(l=150, r=20, t=80, b=20),
                 title="Fit by variable (no combined score)")
hm.show()

## Reading the results responsibly
* Only a few ranges in the CSV are DOA-verified (see the count in step 2). Hatched bars are general-agronomy estimates.
* Rainfall is compared with one season window for all crops; each crop's real growing window may differ.
* pH, soil type and drainage cannot be taken from NASA data. Use soil tests and local knowledge.
* Treat this as a screening aid, then check with local agricultural officers.